In [1]:
import pandas as pd
import random
from transformers import AutoTokenizer
from utils.load import LoadDataCSV
from utils.embeddings import Embeddings

# Load datasets

df_loader=LoadDataCSV()


/Users/santiagolares/Projects/semeval_2025/tformervenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#  Data Preparation

In [2]:
fact_checks_mono_sample_1000=pd.read_csv('data/fact_checks_mono_sample_1000.csv')
posts_mono_sample_1000=pd.read_csv('data/posts_train_mono_sample_1000.csv')

embeddings_utils=Embeddings

embedding_columns_posts = ['ocr_text_embedding', 'ocr_text_eng_embedding']
embedding_columns_fact_checks = ['claim_title_eng_embedding', 'claim_title_embedding']

fact_checks_mono_sample_1000=embeddings_utils.restore_embeddings(fact_checks_mono_sample_1000, embedding_columns_fact_checks)
posts_mono_sample_1000=embeddings_utils.restore_embeddings(posts_mono_sample_1000, embedding_columns_posts)

In [3]:

# Load your data

df_fact_checks, df_posts, df_fact_check_post_mapping = df_loader.load_data()

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-small")


In [4]:
# Create a df of mapping for post_id that show up in posts_mono_sample_1000
mapping_sample_1000=df_fact_check_post_mapping[df_fact_check_post_mapping['post_id'].isin(posts_mono_sample_1000['post_id'])]

In [5]:
# Generate extremely small samples
posts_mono_sample_1000_10=posts_mono_sample_1000.sample(n=10)

# Create a df of mapping for post_id that show up in posts_mono_sample_1000
mapping_sample_1000_10=df_fact_check_post_mapping[df_fact_check_post_mapping['post_id'].isin(posts_mono_sample_1000_10['post_id'])]

fact_checks_mono_sample_1000_10=fact_checks_mono_sample_1000[fact_checks_mono_sample_1000['fact_check_id'].isin(mapping_sample_1000_10['fact_check_id'])]


In [6]:

def split_into_chunks(text, max_tokens=512):
    """
    Divide un texto en chunks de hasta max_tokens tokens.
    """
    if not text or not isinstance(text, str):  # Verifica que sea una cadena de texto
        return []
    tokens = tokenizer.encode(text, truncation=True)
    chunks = [tokens[i:i + max_tokens] for i in range(0, len(tokens), max_tokens)]
    return [tokenizer.decode(chunk, skip_special_tokens=True) for chunk in chunks]

# Chunk posts and fact-check texts
posts_mono_sample_1000["chunks"] = posts_mono_sample_1000["ocr_text"].apply(split_into_chunks)

fact_checks_mono_sample_1000["chunks"] = fact_checks_mono_sample_1000["claim_title"].apply(split_into_chunks)

# Prepare positive and negative pairs
import random

def create_pairs(pairs_df, neg_ratio=5):
    """
    Creates positive and negative pairs for contrastive learning.
    """
    positive_pairs = []
    negative_pairs = []

    all_fact_check_ids = set(pairs_df["fact_check_id"].tolist())
    for _, row in pairs_df.iterrows():
        # Positive pair
        positive_pairs.append((row["post_id"], row["fact_check_id"], 1))
        
        # Negative pairs
        possible_negatives = list(all_fact_check_ids - {row["fact_check_id"]})
        if len(possible_negatives) < neg_ratio:
            neg_ratio = len(possible_negatives)  # Adjust if not enough negatives
        negatives = random.sample(possible_negatives, neg_ratio)
        for neg in negatives:
            negative_pairs.append((row["post_id"], neg, 0))
    
    return positive_pairs + negative_pairs

pairs_data = create_pairs(mapping_sample_1000)


# Fine-Tuning Dataset Class

In [7]:
import torch
from torch.utils.data import Dataset

class FineTuneDataset(Dataset):
    def __init__(self, pairs_data, posts, fact_checks, tokenizer, max_tokens=512):
        self.pairs_data = pairs_data
        self.posts = posts.set_index("post_id")
        self.fact_checks = fact_checks.set_index("fact_check_id")
        self.tokenizer = tokenizer
        self.max_tokens = max_tokens

    def __len__(self):
        return len(self.pairs_data)

    def __getitem__(self, idx):
        post_id, fact_check_id, label = self.pairs_data[idx]

        # Aggregate embeddings for chunked text
        post_chunks = self.posts.loc[post_id]["chunks"]
        # Changed chunk for claim_title, and then back again
        fact_check_chunks = self.fact_checks.loc[fact_check_id]["chunks"]

        post_text = " ".join(post_chunks)
        fact_check_text = " ".join(fact_check_chunks)

        inputs = self.tokenizer(post_text, fact_check_text, truncation=True, 
                                max_length=self.max_tokens, padding="max_length", return_tensors="pt")
        
        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "label": torch.tensor(label, dtype=torch.float)
        }

# Initialize dataset
dataset = FineTuneDataset(pairs_data, posts_mono_sample_1000, fact_checks_mono_sample_1000, tokenizer)


# Model Fine-Tuning

In [8]:
from transformers import AutoModel, AdamW
import torch.nn as nn

class E5FineTuner(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.model = AutoModel.from_pretrained(model_name)
        self.projection = nn.Linear(384, 128)  # Optional projection layer

    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        embeddings = self.projection(outputs.pooler_output)  # Apply projection
        return embeddings


# Training Loop

In [ ]:
from transformers import get_scheduler
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

# Inicializa el escritor de TensorBoard
writer = SummaryWriter(log_dir='./runs/e5_small_1000_1e_5')

torch.mps.empty_cache()

# Hyperparameters
batch_size = 4
epochs = 3
learning_rate = 1e-5

# Prepare DataLoader
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Initialize Model and Optimizer
model = E5FineTuner("intfloat/multilingual-e5-small")
optimizer = AdamW(model.parameters(), lr=learning_rate)

# Scheduler
num_training_steps = len(dataloader) * epochs
scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

# Contrastive Loss
criterion = nn.CosineEmbeddingLoss()

# Training Loop
device = "mps" if torch.backends.mps.is_available() else "cpu"
model.to(device)

for epoch in range(epochs):
    model.train()
    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        
        embeddings = model(input_ids, attention_mask)
        pos_neg_labels = torch.where(labels == 1, torch.tensor(1.0), torch.tensor(-1.0)).to(device)
        

        # Ensure embeddings are split correctly
        half_dim = embeddings.size(1) // 2
        loss = criterion(embeddings[:, :half_dim], embeddings[:, half_dim:], pos_neg_labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        writer.add_scalar('Loss/train', loss.item(), epoch)

        del input_ids, attention_mask, labels, embeddings, pos_neg_labels  # Libera memoria explícitamente
        torch.mps.empty_cache()

        print(f"Epoch {epoch} Loss: {loss.item()}")

    writer.add_scalar('Learning Rate', scheduler.get_lr()[0], epoch)

writer.close()

/Users/santiagolares/Projects/semeval_2025/tformervenv/lib/python3.12/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 0 Loss: 0.1433926820755005
Epoch 0 Loss: 0.21568572521209717
Epoch 0 Loss: 0.26516789197921753
Epoch 0 Loss: 0.19318950176239014
Epoch 0 Loss: 0.09299353510141373
Epoch 0 Loss: 0.1397259384393692
Epoch 0 Loss: 0.06789856404066086
Epoch 0 Loss: 0.12887166440486908
Epoch 0 Loss: 0.19901354610919952
Epoch 0 Loss: 0.13818702101707458
Epoch 0 Loss: 0.07043253630399704
Epoch 0 Loss: 0.07062731683254242
Epoch 0 Loss: 0.2130437046289444
Epoch 0 Loss: 0.34274497628211975
Epoch 0 Loss: 0.27976202964782715
Epoch 0 Loss: 0.13493996858596802
Epoch 0 Loss: 0.3288900852203369
Epoch 0 Loss: 0.0
Epoch 0 Loss: 0.33174848556518555
Epoch 0 Loss: 0.16321918368339539
Epoch 0 Loss: 0.1431269496679306
Epoch 0 Loss: 0.3269100487232208
Epoch 0 Loss: 0.13845059275627136
Epoch 0 Loss: 0.1424894481897354


KeyboardInterrupt: 

In [ ]:
# Guardar el modelo completo (incluyendo configuraciones y optimizador)
torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'loss': loss,
}, "fine_tuned_model_e5_small_1000_1e_5.pth")